<a href="https://colab.research.google.com/github/davidequattrocchi10/perception-engineering/blob/main/sessions/session_01_camera_models_calibration_homography.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 01 — Camera Models, Calibration & Homography
**CV for Robotics Perception | Learning Series**

---

## Why this matters for humanoid robotics

A humanoid robot perceives the world almost entirely through cameras. Before the robot can grasp an object, navigate a room, or identify a person, it must answer a deceptively simple question: *"given a pixel in this image, where is the corresponding point in the real 3D world?"*

That question is answered by **camera geometry**. Every depth estimator, every 6DoF pose estimator, every visual odometry system is built on top of what you will learn in this notebook.

---

## What you will learn
1. The **pinhole camera model** — how 3D points project onto a 2D image
2. **Intrinsic matrix K** — the camera's internal geometry
3. **Lens distortion** — why real cameras don't behave like ideal pinholes
4. **Camera calibration** — how to compute K and distortion coefficients from images
5. **Homography** — how to map one plane to another (used in AR, robotics, visual odometry)

---

## Interview relevance
> *"Explain the pinhole camera model and what the intrinsic matrix represents."*  
> *"How would you calibrate a camera? What are the outputs?"*  
> *"What is a homography and when would you use it?"*

These are **standard junior-to-mid level interview questions** at robotics and perception companies. After this notebook you will be able to answer all three confidently.


---
## 0. Setup
Install and import everything we need. In Colab, OpenCV is pre-installed. We also use NumPy and Matplotlib.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from mpl_toolkits.mplot3d import Axes3D

print(f'OpenCV version: {cv2.__version__}')
print(f'NumPy version:  {np.__version__}')

# Helper: display one or more images side by side in a notebook
def show_images(images, titles, figsize=(14, 5), cmap=None):
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        if len(img.shape) == 2 or cmap == 'gray':
            ax.imshow(img, cmap='gray')
        else:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title, fontsize=12)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

---
## 1. The Pinhole Camera Model

### The core idea

Imagine light rays from the world passing through a single point (the **optical center** or **camera center**) and hitting an **image plane** behind it. This is the pinhole model.

The mathematical relationship between a 3D world point **P = (X, Y, Z)** and its 2D image projection **p = (u, v)** is:

$$u = f_x \cdot \frac{X}{Z} + c_x$$
$$v = f_y \cdot \frac{Y}{Z} + c_y$$

Where:
- **f_x, f_y** = focal lengths in pixels (how "zoomed in" the camera is)
- **c_x, c_y** = principal point (where the optical axis hits the image, usually near the center)
- **Z** = depth of the point (distance from camera along optical axis)

### The Intrinsic Matrix K

These parameters are packed into a 3x3 matrix called **K** (the intrinsic matrix or camera matrix):

$$K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$$

The full projection in homogeneous coordinates:
$$\lambda \begin{bmatrix} u \\ v \\ 1 \end{bmatrix} = K \begin{bmatrix} X \\ Y \\ Z \end{bmatrix}$$

Where λ = Z (the depth, used for normalization).

**K is unique to each camera** — it depends on the lens, sensor size, and resolution. It does NOT change when you move or rotate the camera (that's handled by extrinsics). Knowing K is essential for any 3D reconstruction or depth estimation task.

In [ ]:
# ── 1.1  Manually simulate the pinhole projection ──────────────────────────
#
# We define a synthetic K (typical for a 640x480 camera) and project
# a set of 3D points onto the image plane BY HAND so you can see
# exactly what the math does.

# Image resolution
W, H = 640, 480

# Define the intrinsic matrix K
# fx = fy ≈ W means ~90° horizontal field of view (common for robotics cameras)
fx, fy = 600.0, 600.0   # focal lengths in pixels
cx, cy = W / 2, H / 2   # principal point at image center

K = np.array([
    [fx,  0, cx],
    [ 0, fy, cy],
    [ 0,  0,  1]
], dtype=np.float64)

print('Intrinsic matrix K:')
print(K)

# ── 1.2  Define some 3D points in camera space ─────────────────────────────
# Each row is one point: [X, Y, Z]
# Z > 0 means in front of the camera
# Think of these as corners of an object the robot is looking at
points_3d = np.array([
    [ 0.0,  0.0, 2.0],   # directly ahead, 2 m away  → should land at (cx, cy)
    [ 0.5,  0.0, 2.0],   # 0.5 m to the right
    [-0.5,  0.0, 2.0],   # 0.5 m to the left
    [ 0.0,  0.5, 2.0],   # 0.5 m below  (Y↓ in camera convention)
    [ 0.0, -0.5, 2.0],   # 0.5 m above
    [ 0.3,  0.3, 1.0],   # closer object — should project FURTHER from center
    [ 0.3,  0.3, 4.0],   # same angle but twice as far — closer to center
], dtype=np.float64)

# ── 1.3  Project using the formula: (u,v) = (fx*X/Z + cx,  fy*Y/Z + cy) ───
def project_points(pts_3d, K):
    """
    Project Nx3 array of 3D points (camera frame) onto the image plane.
    Returns Nx2 array of pixel coordinates.
    """
    X, Y, Z = pts_3d[:, 0], pts_3d[:, 1], pts_3d[:, 2]
    u = K[0, 0] * (X / Z) + K[0, 2]   # fx * X/Z + cx
    v = K[1, 1] * (Y / Z) + K[1, 2]   # fy * Y/Z + cy
    return np.stack([u, v], axis=1)

pixels = project_points(points_3d, K)

print('\n3D point (X,Y,Z)  →  pixel (u, v)')
for p3, p2 in zip(points_3d, pixels):
    print(f'  {p3}  →  ({p2[0]:.1f}, {p2[1]:.1f})')

In [ ]:
# ── 1.4  Visualise where the 3D points land on the image plane ─────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── LEFT: 3D scene (top-down view) ─────────────────────────────────────────
ax3d = fig.add_subplot(121, projection='3d')
ax3d.scatter(points_3d[:, 0], points_3d[:, 2], -points_3d[:, 1],
             c='royalblue', s=80, zorder=5)
ax3d.scatter(0, 0, 0, c='red', s=120, marker='^', label='Camera')
for i, p in enumerate(points_3d):
    ax3d.text(p[0], p[2], -p[1] + 0.05, f'P{i}', fontsize=8)
ax3d.set_xlabel('X (right)')
ax3d.set_ylabel('Z (depth)')
ax3d.set_zlabel('Y (up)')
ax3d.set_title('3D Scene (Camera at origin)')
ax3d.legend()

# ── RIGHT: image plane with projected points ────────────────────────────────
axes[1].set_xlim(0, W)
axes[1].set_ylim(H, 0)          # image Y goes downward
axes[1].set_facecolor('#1a1a2e')
axes[1].set_title('Projected onto Image Plane (640×480)', color='white')
axes[1].tick_params(colors='white')
for spine in axes[1].spines.values():
    spine.set_edgecolor('white')

# Draw principal point
axes[1].scatter(cx, cy, c='red', s=100, zorder=5, label=f'Principal point ({cx:.0f},{cy:.0f})')
axes[1].axhline(cy, color='red', linestyle='--', alpha=0.3)
axes[1].axvline(cx, color='red', linestyle='--', alpha=0.3)

# Draw projected points
colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(pixels)))
for i, (px, col) in enumerate(zip(pixels, colors)):
    axes[1].scatter(px[0], px[1], color=col, s=120, zorder=6)
    axes[1].annotate(f'P{i} (Z={points_3d[i,2]}m)',
                     (px[0], px[1]),
                     textcoords='offset points', xytext=(8, 4),
                     color=col, fontsize=9)

axes[1].legend(loc='lower right', facecolor='#1a1a2e', labelcolor='white')

plt.tight_layout()
plt.suptitle('Pinhole Camera Model: 3D → 2D Projection', y=1.01,
             fontsize=14, fontweight='bold')
plt.show()

print('\nObservations to notice:')
print('  • P0 (directly ahead) lands exactly at the principal point (320, 240)')
print('  • P5 and P6 are the same direction but different depths.')
print('    P5 (Z=1m) is projected further from center than P6 (Z=4m).')
print('    This is perspective foreshortening — depth is lost in projection.')

---
## 2. Lens Distortion

Real cameras don't behave like ideal pinholes. Glass lenses introduce **distortion** that bends straight lines in the world into curves in the image. There are two main types:

| Type | Effect | Typical cause |
|---|---|---|
| **Radial distortion** | Lines bow outward (barrel) or inward (pincushion) from the image center | Spherical lens shape |
| **Tangential distortion** | Image appears slightly tilted/shifted | Lens not perfectly parallel to sensor |

OpenCV models distortion with 5 coefficients: **(k1, k2, p1, p2, k3)**
- k1, k2, k3 = radial coefficients
- p1, p2 = tangential coefficients

The corrected coordinates **(x_corrected, y_corrected)** from normalized coordinates **(x, y)** are:
$$r^2 = x^2 + y^2$$
$$x_{dist} = x(1 + k_1 r^2 + k_2 r^4 + k_3 r^6) + 2p_1 xy + p_2(r^2 + 2x^2)$$

**Why does this matter for robotics?** If you run object detection or pose estimation on a distorted image, your results will be geometrically wrong. Every real robotics perception pipeline undistorts images first.

In [ ]:
# ── 2.1  Simulate radial distortion on a synthetic grid ────────────────────
#
# We generate a clean grid image (what an ideal pinhole would see)
# and then apply barrel distortion to show what a real wide-angle lens does.

def make_grid_image(width=640, height=480, spacing=40):
    """Create a perfect grid image — straight lines in the ideal world."""
    img = np.ones((height, width, 3), dtype=np.uint8) * 240
    for x in range(0, width, spacing):
        cv2.line(img, (x, 0), (x, height), (80, 80, 200), 1)
    for y in range(0, height, spacing):
        cv2.line(img, (0, y), (width, y), (80, 80, 200), 1)
    # Add a few reference circles
    cv2.circle(img, (width // 2, height // 2), 5, (220, 50, 50), -1)
    cv2.circle(img, (width // 4, height // 4), 5, (50, 150, 50), -1)
    cv2.circle(img, (3 * width // 4, 3 * height // 4), 5, (50, 150, 50), -1)
    return img

grid_clean = make_grid_image()

# Define camera K and distortion coefficients (barrel distortion: k1 < 0)
K_dist = np.array([[600., 0., 320.],
                   [  0., 600., 240.],
                   [  0.,   0.,   1.]], dtype=np.float64)

# Strong barrel distortion (negative k1) — typical of wide-angle cameras
dist_barrel  = np.array([-0.4, 0.15, 0.0, 0.0, 0.0])   # barrel
# Pincushion: positive k1
dist_pincush = np.array([ 0.4, 0.0,  0.0, 0.0, 0.0])   # pincushion

h, w = grid_clean.shape[:2]

# Apply distortion using cv2.undistort in reverse:
# We remap the clean image THROUGH the distortion model
def apply_distortion(img, K, dist_coeffs):
    """Warp a clean image to look as if captured through a distorted lens."""
    h, w = img.shape[:2]
    # Build pixel grid
    u_coords, v_coords = np.meshgrid(np.arange(w), np.arange(h))
    # Normalise to camera coordinates
    x_norm = (u_coords - K[0, 2]) / K[0, 0]
    y_norm = (v_coords - K[1, 2]) / K[1, 1]
    r2 = x_norm**2 + y_norm**2
    k1, k2, p1, p2, k3 = dist_coeffs
    # Radial + tangential
    radial = 1 + k1 * r2 + k2 * r2**2 + k3 * r2**3
    x_dist = x_norm * radial + 2 * p1 * x_norm * y_norm + p2 * (r2 + 2 * x_norm**2)
    y_dist = y_norm * radial + p1 * (r2 + 2 * y_norm**2) + 2 * p2 * x_norm * y_norm
    # Back to pixel coordinates
    u_src = (x_dist * K[0, 0] + K[0, 2]).astype(np.float32)
    v_src = (y_dist * K[1, 1] + K[1, 2]).astype(np.float32)
    return cv2.remap(img, u_src, v_src, cv2.INTER_LINEAR,
                     borderMode=cv2.BORDER_CONSTANT, borderValue=(255, 255, 255))

grid_barrel  = apply_distortion(grid_clean, K_dist, dist_barrel)
grid_pincush = apply_distortion(grid_clean, K_dist, dist_pincush)

show_images(
    [grid_clean, grid_barrel, grid_pincush],
    ['Ideal (no distortion)', 'Barrel distortion (k1=-0.4)\nCommon in wide-angle/robot cams',
     'Pincushion distortion (k1=+0.4)\nCommon in telephoto lenses'],
    figsize=(16, 5)
)

print('Notice how the straight grid lines CURVE near the edges.')
print('In barrel distortion, lines bow OUTWARD (away from center).')
print('In pincushion, lines bow INWARD (toward center).')

In [ ]:
# ── 2.2  Undistort: recover the clean image ─────────────────────────────────
#
# cv2.undistort() takes the distorted image + K + distortion coefficients
# and returns the corrected image. This is one of the most-used functions
# in any real robotics perception pipeline.

grid_undistorted = cv2.undistort(grid_barrel, K_dist, dist_barrel)

show_images(
    [grid_barrel, grid_undistorted],
    ['Distorted (barrel)', 'After cv2.undistort() — lines are straight again'],
    figsize=(12, 5)
)

# ── 2.3  Optimal new camera matrix ─────────────────────────────────────────
#
# After undistortion, some pixels near the border are undefined (black).
# getOptimalNewCameraMatrix lets you control the trade-off:
#   alpha=0 → crop to keep only valid pixels (no black border)
#   alpha=1 → keep all pixels (some black border at edges)

new_K, roi = cv2.getOptimalNewCameraMatrix(
    K_dist, dist_barrel, (w, h), alpha=0
)
grid_optimal = cv2.undistort(grid_barrel, K_dist, dist_barrel, newCameraMatrix=new_K)

print('Original K focal length: fx =', K_dist[0, 0])
print('Optimal  K focal length: fx =', new_K[0, 0])
print('ROI (region of valid pixels):', roi)
print()
print('The optimal K has a slightly different focal length because the')
print('undistortion process effectively rescales the image.')

---
## 3. Camera Calibration

In the previous sections we assumed we already *knew* K and the distortion coefficients. In practice you have to **measure** them from images. This is **camera calibration**.

### The algorithm (Zhang's method)

OpenCV implements Zhang's method (2000). The idea:
1. Print a **chessboard pattern** (known geometry — we know the exact size of each square)
2. Take **10–30 photos** of it from different angles and distances
3. OpenCV **detects the corners** in each image (sub-pixel accuracy)
4. Using the known 3D positions of those corners and their 2D positions in the image, solve for **K** and **distortion coefficients** via least squares

### Outputs of calibration
- **K** (3×3 intrinsic matrix) — never changes for this camera+lens combo
- **dist** (5 distortion coefficients)
- **rvecs, tvecs** — rotation and translation of each calibration image (extrinsics — useful for debugging but discarded after)
- **reprojection error** — the key quality metric; < 1.0 pixel is good, < 0.5 is excellent

> **Interview tip:** Always mention reprojection error when discussing calibration. It shows you know how to evaluate calibration quality, not just run the function.

In [ ]:
# ── 3.1  Generate synthetic calibration images ─────────────────────────────
#
# In a real scenario you would take photos with your camera.
# Here we SIMULATE the process: we define a ground-truth K and distortion,
# then generate synthetic chessboard images and try to recover K from them.
# This is the cleanest way to validate you understand the pipeline.

# Ground truth camera parameters (what we want to recover)
K_true = np.array([[800., 0., 320.],
                   [  0., 800., 240.],
                   [  0.,   0.,   1.]], dtype=np.float64)
dist_true = np.array([-0.2, 0.05, 0.001, 0.001, 0.0])

# Chessboard parameters
BOARD_W  = 9    # inner corners along width
BOARD_H  = 6    # inner corners along height
SQ_SIZE  = 0.025  # real-world square size in meters (2.5 cm)
IMG_W, IMG_H = 640, 480

# 3D coordinates of chessboard corners in board frame (Z=0 plane)
objp = np.zeros((BOARD_W * BOARD_H, 3), dtype=np.float32)
objp[:, :2] = np.mgrid[0:BOARD_W, 0:BOARD_H].T.reshape(-1, 2)
objp *= SQ_SIZE  # scale to real-world units (meters)

def generate_chessboard_view(K, dist, rvec, tvec, board_size=(9, 6)):
    """
    Generate a synthetic view of the chessboard from a given pose.
    Returns: image, detected 2D corner points
    """
    bw, bh = board_size
    # 3D corners in board frame
    obj = np.zeros((bw * bh, 3), np.float32)
    obj[:, :2] = np.mgrid[0:bw, 0:bh].T.reshape(-1, 2) * SQ_SIZE

    # Project 3D corners → 2D pixels using the true camera model
    imgpts, _ = cv2.projectPoints(obj, rvec, tvec, K, dist)
    imgpts = imgpts.reshape(-1, 2)

    # Build image
    img = np.ones((IMG_H, IMG_W, 3), dtype=np.uint8) * 255
    for r in range(bh - 1):
        for c in range(bw - 1):
            idx = r * bw + c
            corners = imgpts[[idx, idx + 1, idx + bw + 1, idx + bw]].astype(np.int32)
            if (r + c) % 2 == 0:
                cv2.fillPoly(img, [corners], (30, 30, 30))

    return img, imgpts.astype(np.float32).reshape(-1, 1, 2)

# Define 20 different camera poses (random rotations/translations viewing the board)
np.random.seed(42)
n_images = 20
obj_points = []   # 3D corner positions (same for every image — always the same board)
img_points = []   # 2D corner projections per image
sample_images = []

for i in range(n_images):
    # Random small rotations and translations around a frontal view
    rx = np.random.uniform(-0.4, 0.4)
    ry = np.random.uniform(-0.4, 0.4)
    rz = np.random.uniform(-0.1, 0.1)
    tx = np.random.uniform(-0.05, 0.05)
    ty = np.random.uniform(-0.03, 0.03)
    tz = np.random.uniform(0.30, 0.60)   # board 30–60 cm from camera

    rvec = np.array([rx, ry, rz], dtype=np.float64)
    tvec = np.array([tx, ty, tz], dtype=np.float64)

    img, corners = generate_chessboard_view(K_true, dist_true, rvec, tvec)

    # Only keep images where all corners project inside the image
    if corners is not None and len(corners) == BOARD_W * BOARD_H:
        pts = corners.reshape(-1, 2)
        if (pts[:, 0].min() > 5 and pts[:, 0].max() < IMG_W - 5 and
            pts[:, 1].min() > 5 and pts[:, 1].max() < IMG_H - 5):
            obj_points.append(objp)
            img_points.append(corners)
            if len(sample_images) < 4:
                sample_images.append((img, corners))

print(f'Valid calibration images: {len(obj_points)} / {n_images}')

In [ ]:
# ── 3.2  Visualise sample calibration images with detected corners ──────────

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (img, corners) in zip(axes, sample_images):
    vis = img.copy()
    cv2.drawChessboardCorners(vis, (BOARD_W, BOARD_H), corners, True)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.axis('off')
plt.suptitle('Sample Synthetic Calibration Images\n(corners detected in colour, ordered from first to last)',
             fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.3  Run calibration and evaluate ──────────────────────────────────────

ret, K_estimated, dist_estimated, rvecs, tvecs = cv2.calibrateCamera(
    obj_points, img_points, (IMG_W, IMG_H), None, None
)

print('═' * 55)
print(' CALIBRATION RESULTS')
print('═' * 55)
print(f'\nRMS Reprojection Error: {ret:.4f} pixels')
print('  < 0.5 px  → excellent')
print('  < 1.0 px  → good (acceptable for most robotics applications)')
print('  > 1.5 px  → poor (recalibrate)')

print('\n── Intrinsic Matrix K ──────────────────────────────')
print('  Ground truth:')
print(f'    fx={K_true[0,0]:.1f}, fy={K_true[1,1]:.1f}, cx={K_true[0,2]:.1f}, cy={K_true[1,2]:.1f}')
print('  Estimated:')
print(f'    fx={K_estimated[0,0]:.2f}, fy={K_estimated[1,1]:.2f}, '
      f'cx={K_estimated[0,2]:.2f}, cy={K_estimated[1,2]:.2f}')
print(f'  Error: fx={abs(K_true[0,0]-K_estimated[0,0]):.3f} px, '
      f'fy={abs(K_true[1,1]-K_estimated[1,1]):.3f} px')

print('\n── Distortion Coefficients (k1, k2, p1, p2, k3) ───')
print('  Ground truth:', dist_true)
print('  Estimated:   ', np.round(dist_estimated.ravel(), 5))

# ── 3.4  Reprojection error per image (quality check) ──────────────────────
per_image_errors = []
for i in range(len(obj_points)):
    reproj, _ = cv2.projectPoints(obj_points[i], rvecs[i], tvecs[i],
                                   K_estimated, dist_estimated)
    err = cv2.norm(img_points[i], reproj, cv2.NORM_L2) / np.sqrt(len(reproj))
    per_image_errors.append(err)

fig, ax = plt.subplots(figsize=(10, 3))
bars = ax.bar(range(len(per_image_errors)), per_image_errors,
              color=['#e74c3c' if e > 1.0 else '#2ecc71' for e in per_image_errors])
ax.axhline(1.0, color='orange', linestyle='--', label='1.0 px threshold')
ax.axhline(0.5, color='green',  linestyle='--', alpha=0.6, label='0.5 px threshold')
ax.set_xlabel('Image index')
ax.set_ylabel('Reprojection error (px)')
ax.set_title('Per-image Reprojection Error\n(green bars < 1px, red bars > 1px → consider removing)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nMean per-image error: {np.mean(per_image_errors):.4f} px')

---
## 4. Homography

### What is a homography?

A **homography** is a 3×3 matrix **H** that maps points from one **plane** to another. It describes the relationship between two views of the same planar surface.

$$\lambda \begin{bmatrix} u' \\ v' \\ 1 \end{bmatrix} = H \begin{bmatrix} u \\ v \\ 1 \end{bmatrix}$$

### Why does it matter for robotics?

| Use case | How homography is used |
|---|---|
| **Bird's-eye view (BEV)** | Warp a ground-facing camera to a top-down map view |
| **Augmented Reality** | Map virtual objects onto a detected flat surface |
| **Visual odometry** | Estimate camera motion between frames (for planar scenes) |
| **Image stitching / mosaics** | Align multiple images of a planar scene |
| **Marker detection** | Align a detected ArUco/AprilTag to a known template |

You need only **4 point correspondences** (non-collinear) to compute a homography.

> **Interview question:** *"How would you compute a bird's-eye view of a road from a front-facing camera?"*  
> **Answer:** Use a homography. Select 4 points on the road plane, define their desired positions in the top-down view, compute H with `cv2.findHomography`, and warp with `cv2.warpPerspective`.

In [ ]:
# ── 4.1  Bird's-eye view (BEV) via homography ──────────────────────────────
#
# We simulate a road scene viewed from a front-facing camera mounted on
# a robot/car. The road markings appear in perspective.
# We use a homography to get a top-down (bird's-eye) view — used in
# autonomous driving and mobile robotics for obstacle mapping.

def make_road_image(width=640, height=480):
    """Simulate a road viewed from a front-facing camera (perspective)."""
    img = np.ones((height, width, 3), dtype=np.uint8)
    img[:] = (80, 80, 80)   # asphalt

    # Horizon
    horizon = height // 2
    img[:horizon] = (135, 180, 215)   # sky

    # Road edges (trapezoid in perspective)
    road_pts = np.array([[160, height], [480, height],
                         [360, horizon + 20], [280, horizon + 20]])
    cv2.fillPoly(img, [road_pts], (100, 100, 100))

    # Lane markings (dashed centre line in perspective)
    for y_far, y_near in [(horizon + 25, horizon + 55),
                           (horizon + 65, horizon + 120),
                           (horizon + 135, horizon + 230),
                           (horizon + 245, height - 20)]:
        # Interpolate x positions based on depth
        t_far  = (y_far  - horizon) / (height - horizon)
        t_near = (y_near - horizon) / (height - horizon)
        x_far  = int(width // 2 - 5  * t_far)
        x_near = int(width // 2 - 10 * t_near)
        w_far  = max(2, int(4  * t_far))
        w_near = max(2, int(10 * t_near))
        pts = np.array([[x_far - w_far, y_far], [x_far + w_far, y_far],
                         [x_near + w_near, y_near], [x_near - w_near, y_near]])
        cv2.fillPoly(img, [pts], (255, 255, 200))

    return img

road = make_road_image()

# ── 4.2  Define source points (on road plane in perspective image) ──────────
# These are the 4 corners of a rectangle on the road surface that we
# know is actually a rectangle in the real world.
H, W = road.shape[:2]
horizon = H // 2

# Points in the perspective image (bottom trapezoid of the road)
src_pts = np.float32([
    [280, horizon + 20],   # top-left of road region
    [360, horizon + 20],   # top-right
    [480, H - 10],         # bottom-right
    [160, H - 10]          # bottom-left
])

# Destination points — where those same points land in the top-down view
# We map the trapezoid to a rectangle
dst_pts = np.float32([
    [160, 10],             # top-left
    [480, 10],             # top-right
    [480, H - 10],         # bottom-right
    [160, H - 10]          # bottom-left
])

# ── 4.3  Compute homography matrix H ───────────────────────────────────────
# findHomography uses RANSAC internally if you have more than 4 points
# (robust to outliers). With exactly 4 points it solves directly.
Hmat, mask = cv2.findHomography(src_pts, dst_pts, method=0)

print('Homography matrix H:')
print(np.round(Hmat, 4))
print('\nH has 8 degrees of freedom (9 entries, scale-invariant).')
print('It encodes rotation, translation, scale, AND perspective distortion.')

# ── 4.4  Warp the image ─────────────────────────────────────────────────────
bev = cv2.warpPerspective(road, Hmat, (W, H))

# Draw the source region on the original image
road_annotated = road.copy()
cv2.polylines(road_annotated, [src_pts.astype(np.int32)], True, (0, 255, 0), 2)
for pt in src_pts:
    cv2.circle(road_annotated, tuple(pt.astype(int)), 6, (0, 0, 255), -1)

show_images(
    [road_annotated, bev],
    ['Front-facing camera (perspective)\nGreen region = road plane used for homography',
     "Bird's-eye view after warpPerspective(H)\nLane markings are now parallel"],
    figsize=(14, 6)
)

In [ ]:
# ── 4.5  Understand H: decompose it manually ────────────────────────────────
#
# Let's apply H to our src_pts and verify they land on dst_pts.
# This is how you verify a homography is correct.

def apply_homography(H, points):
    """
    Apply homography H to an Nx2 array of points.
    Uses homogeneous coordinates: [x, y, 1] → H @ [x, y, 1] → normalise.
    """
    # Convert to homogeneous: Nx3
    ones = np.ones((len(points), 1), dtype=np.float32)
    pts_h = np.hstack([points, ones])   # Nx3
    # Apply H: result is Nx3
    result_h = (H @ pts_h.T).T          # Nx3
    # Normalise by third coordinate (lambda)
    result = result_h[:, :2] / result_h[:, 2:3]
    return result

src_transformed = apply_homography(Hmat, src_pts)

print('Verification — src_pts transformed by H should equal dst_pts:')
print(f'{"Source":>40}  →  {"H @ source":>20}  ==  {"Destination":>20}')
for s, t, d in zip(src_pts, src_transformed, dst_pts):
    match = '✓' if np.allclose(t, d, atol=0.5) else '✗'
    print(f'  ({s[0]:5.0f}, {s[1]:5.0f})  →  ({t[0]:7.2f}, {t[1]:7.2f})  '
          f'== ({d[0]:5.0f}, {d[1]:5.0f})  {match}')

---
## 5. Putting it all together — The Perception Pipeline

Every time a real robotics perception system processes a frame, it goes through this chain:

```
Raw camera frame
       │
       ▼
cv2.undistort(frame, K, dist)          ← use calibrated K and dist
       │
       ▼
Perception algorithm                   ← detection, segmentation, depth...
       │
       ▼
3D reasoning using K                   ← back-project pixel → 3D ray
       │
       ▼
Action / control
```

Let's implement the **back-projection** step: given a pixel (u, v) and a known depth Z, recover the 3D point.

In [ ]:
# ── 5.1  Back-projection: pixel + depth → 3D point ─────────────────────────
#
# This is the INVERSE of the projection we did in Section 1.
# Given pixel (u, v) and depth Z (from a depth camera like RealSense or D435),
# recover 3D point (X, Y, Z) in camera frame.
#
# From the projection equations:
#   u = fx * X/Z + cx  →  X = (u - cx) * Z / fx
#   v = fy * Y/Z + cy  →  Y = (v - cy) * Z / fy

def backproject(u, v, depth_m, K):
    """
    Back-project a pixel (u, v) at known depth Z into 3D camera coordinates.

    Args:
        u, v    : pixel coordinates
        depth_m : depth in meters (from depth sensor)
        K       : 3x3 intrinsic matrix

    Returns:
        (X, Y, Z) in camera frame (meters)
    """
    fx, fy = K[0, 0], K[1, 1]
    cx, cy = K[0, 2], K[1, 2]
    X = (u - cx) * depth_m / fx
    Y = (v - cy) * depth_m / fy
    Z = depth_m
    return np.array([X, Y, Z])

# Use our calibrated K
K_cam = K_estimated

# Simulate: the robot detects an object at pixel (350, 290) using object detection.
# The depth camera reads 1.5 m at that pixel.
u_detect, v_detect = 350, 290
depth_at_pixel = 1.5   # meters

point_3d = backproject(u_detect, v_detect, depth_at_pixel, K_cam)

print('── Back-projection example ───────────────────────────────────────────')
print(f'Pixel detected at: ({u_detect}, {v_detect})')
print(f'Depth from sensor: {depth_at_pixel} m')
print(f'3D point in camera frame: X={point_3d[0]:.4f} m, Y={point_3d[1]:.4f} m, Z={point_3d[2]:.4f} m')
print()
print('This tells the robot: "the object is 1.5 m ahead, slightly to the right')
print(f'(X={point_3d[0]:.3f} m), and slightly below centre (Y={point_3d[1]:.3f} m)."')
print()
print('The robot arm / navigation planner then uses these 3D coordinates.')

# ── 5.2  Back-project the FULL depth map (as a depth camera gives you) ──────
print('\n── Batch back-projection (full depth map) ───────────────────────────')

# Simulate a synthetic depth map (closer in the centre, farther at edges)
u_grid, v_grid = np.meshgrid(np.arange(IMG_W), np.arange(IMG_H))
depth_map = 2.0 + 0.5 * ((u_grid - IMG_W/2)**2 + (v_grid - IMG_H/2)**2) / (IMG_W**2)

# Vectorised back-projection — same formula, applied to entire grid at once
X_map = (u_grid - K_cam[0, 2]) * depth_map / K_cam[0, 0]
Y_map = (v_grid - K_cam[1, 2]) * depth_map / K_cam[1, 1]
Z_map = depth_map

print(f'Point cloud shape: {X_map.shape}  ({IMG_W * IMG_H:,} 3D points)')
print(f'X range: [{X_map.min():.2f}, {X_map.max():.2f}] m')
print(f'Y range: [{Y_map.min():.2f}, {Y_map.max():.2f}] m')
print(f'Z range: [{Z_map.min():.2f}, {Z_map.max():.2f}] m')
print()
print('Every pixel of a depth camera gives you one 3D point.')
print('Stack all of them → you have a POINT CLOUD, the input to 3D perception.')

---
## Summary — What you learned today

| Concept | What it is | Why it matters for robotics |
|---|---|---|
| **Pinhole model** | 3D → 2D projection via K | Foundation of all geometric CV |
| **Intrinsic matrix K** | fx, fy, cx, cy | Required for any 3D reasoning from images |
| **Lens distortion** | k1,k2,p1,p2,k3 | Real cameras need undistortion before processing |
| **Camera calibration** | Estimate K + dist from images | Must know how to do this in any lab or field deployment |
| **Reprojection error** | Quality metric of calibration | < 1.0 px acceptable, < 0.5 px excellent |
| **Homography** | Planar 2D→2D mapping | BEV, AR, visual odometry, marker tracking |
| **Back-projection** | pixel + depth → 3D point | How the robot knows WHERE objects are in space |

---

## Interview cheat sheet

**Q: What does the intrinsic matrix K encode?**  
A: The camera's internal geometry — focal lengths (fx, fy) and principal point (cx, cy). It is unique to a camera+lens combination and does not change with camera movement.

**Q: What is camera calibration and what are its outputs?**  
A: The process of estimating K and distortion coefficients from multiple views of a known pattern (chessboard). Outputs: K, dist, and per-image extrinsics. Quality is measured by RMS reprojection error.

**Q: What is a homography and when would you use it?**  
A: A 3×3 matrix mapping points between two views of the same plane. Used for bird's-eye view generation, AR, image stitching, and visual odometry in planar scenes.

**Q: How do you get a 3D position from a 2D detection + depth?**  
A: Back-projection using K: X = (u - cx) × Z / fx, Y = (v - cy) × Z / fy.

---

## Exercises (try these yourself before Session 02)

1. **Change fx**: Set `fx = 300` in Section 1. What happens to the projected points? Why?
2. **Stronger distortion**: Change `k1` to `-0.8` in Section 2. What do you notice?
3. **Calibration with fewer images**: Re-run Section 3 with only 5 images. Does the reprojection error get worse? Why is having diverse viewing angles important?
4. **Homography verification**: In Section 4, change the `dst_pts` so the BEV is zoomed in more. What do you change?
5. **(Challenge)**: Implement a function `reproject(point_3d, K)` that projects a 3D point back to 2D AND verify that `backproject(reproject(P, K), Z, K) ≈ P` (they are inverses).
